# Phase 4 — Sentiment Analysis: Emotion Scoring

We use the Hugging Face model `j-hartmann/emotion-english-distilroberta-base` — a DistilRoBERTa model fine-tuned to classify text into 6 emotions:
- joy, surprise, anger, fear, sadness, disgust (+ neutral)

### Key design decision: sentence-level scoring
Rather than classifying the whole description at once (which can get truncated and averages out emotion peaks), we:
1. Split each description into sentences.
2. Score each sentence independently.
3. Take the **maximum** score per emotion across all sentences.

This captures the *emotional peak* — so a thriller that's mostly dry but has one terrifying sentence will score high on Fear.

**Prerequisites:** `3-text-classification.ipynb` → `data/books_with_categories.csv`

**Output:** `data/books_with_emotions.csv`

In [ ]:
import re
import pandas as pd
from transformers import pipeline

## 1. Load the categorised data

In [ ]:
df = pd.read_csv('../data/books_with_categories.csv')
print(f'Loaded {len(df)} books')
df.columns.tolist()

## 2. Load the emotion classifier

In [ ]:
emotion_classifier = pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    top_k=None   # return all 7 emotion scores, not just the winner
)

# Emotions we'll keep as features (ignoring 'disgust' and 'neutral')
EMOTIONS = ['joy', 'surprise', 'anger', 'fear', 'sadness']

## 3. Test on a single description

In [ ]:
sample_desc = df['description'].iloc[5]
sentences = re.split(r'(?<=[.!?])\s+', sample_desc.strip())
print(f'Description split into {len(sentences)} sentences.')

# Score one sentence
result = emotion_classifier(sentences[0][:512])
print('\nScores for first sentence:')
for item in sorted(result[0], key=lambda x: -x['score']):
    print(f"  {item['label']:12s}: {item['score']:.4f}")

## 4. Build the scoring function

In [ ]:
def get_max_emotion_scores(description: str) -> dict:
    """
    Split description into sentences. Score each sentence with the emotion
    classifier. Return a dict of {emotion: max_score_across_sentences}.
    """
    sentences = re.split(r'(?<=[.!?])\s+', description.strip())
    # Skip fragments that are too short to carry emotion
    sentences = [s for s in sentences if len(s.split()) > 3]

    max_scores = {e: 0.0 for e in EMOTIONS}

    for sentence in sentences:
        try:
            results = emotion_classifier(sentence[:512])  # guard against token limit
            for item in results[0]:
                label = item['label'].lower()
                if label in max_scores:
                    max_scores[label] = max(max_scores[label], item['score'])
        except Exception:
            continue  # skip malformed sentences

    return max_scores

## 5. Score all books

> ⏱ **Expected time:** ~30–60 minutes on CPU. Run once and save.

In [ ]:
print('Scoring emotions across all book descriptions...')
emotion_rows = df['description'].apply(get_max_emotion_scores)
emotion_df = pd.DataFrame(emotion_rows.tolist())

# Title-case column names to match the dashboard's expectations
emotion_df.rename(columns={
    'joy': 'joy',
    'surprise': 'surprise',
    'anger': 'anger',
    'fear': 'fear',
    'sadness': 'sadness'
}, inplace=True)

df = pd.concat([df, emotion_df], axis=1)
print('Emotion columns added.')
df[['title'] + EMOTIONS].sample(5)

## 6. Visualise the emotion distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(EMOTIONS), figsize=(18, 4))
for ax, emotion in zip(axes, EMOTIONS):
    df[emotion].hist(bins=40, ax=ax)
    ax.set_title(emotion.capitalize())
    ax.set_xlabel('Max score')
    ax.set_ylabel('Books')
plt.suptitle('Distribution of peak emotion scores', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 7. Save the fully enriched dataset

In [ ]:
df.to_csv('../data/books_with_emotions.csv', index=False)
print(f'Saved {len(df)} books with emotion scores to data/books_with_emotions.csv')
print('Columns:', df.columns.tolist())